# Watch a model actually learn

**Nothing to install. Press the ▶ button on each cell, top to bottom.**

Everyone calls `.fit()` and a model appears. This notebook opens the box: we'll teach a computer to
predict sales from ad spend using nothing but arithmetic you could do on paper — then watch it walk,
blindfolded, to exactly the answer algebra proves.

Pairs with **ML_Study_01**, sections 3.4–3.7.

**What you'll see:**
1. The loop that *is* training
2. The line improving, plotted step by step
3. A slider — break it yourself by making the steps too big
4. The scikit-learn version, and a surprise most courses get wrong

## 1. The data — four weeks of a small business

Four weeks. What we spent on ads, and what we sold. Both in **thousands of dollars**.

That's the whole dataset. Small enough to check by hand — which is the point.

In [ ]:
# What we spent on ads each week ($1,000s). This is what we KNOW (the input).
ad_spend = [1, 2, 3, 4]

# What we sold each week ($1,000s). This is what we want to PREDICT (the answer).
sales = [5, 7, 8, 11]

# How many weeks of data we have. "m" is the traditional letter for "how many examples".
m = len(ad_spend)

# Draw the four dots so we can see what we're dealing with.
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter(ad_spend, sales, s=90, zorder=3)          # the four weeks, as dots
plt.xlabel("Ad spend ($1,000s)")
plt.ylabel("Weekly sales ($1,000s)")
plt.title("Four weeks. Spend more -> sell more. But by exactly how much?")
plt.grid(alpha=0.3)
plt.show()

print("Our job: draw the single straight line that comes closest to all four dots.")

## 2. The model, the score, and the slopes

Three tiny functions. This is the entire "AI".

- **`predict`** — the model itself: start at `theta0`, add `theta1` for every $1,000 spent.
- **`cost`** — a *badness score*. How far off are we? Square the misses so they can't cancel out. Lower is better, like golf.
- **`slopes`** — which way is *uphill* for each knob, so we know which way to run.

In [ ]:
def predict(theta0, theta1, x):
    """Our whole model: start at theta0, add theta1 for each unit of ad spend."""
    return theta0 + theta1 * x


def cost(theta0, theta1):
    """The badness score. Lower = better. (The /2 is a math convenience - see 3.3.)"""
    total = 0.0
    for x, y in zip(ad_spend, sales):
        error = predict(theta0, theta1, x) - y   # how far off were we this week?
        total += error ** 2                      # square it: kills minus signs, punishes big misses
    return total / (2 * m)                       # average it, so 4 rows or 4 million behaves the same


def slopes(theta0, theta1):
    """Which way is UPHILL for each knob. Both measured from the SAME spot."""
    # The height knob's slope is simply the average error.
    slope0 = sum(predict(theta0, theta1, x) - y for x, y in zip(ad_spend, sales)) / m
    # The steepness knob's slope is the average error, weighted by ad spend.
    slope1 = sum((predict(theta0, theta1, x) - y) * x for x, y in zip(ad_spend, sales)) / m
    return slope0, slope1


# Try a deliberately awful line: flat at zero, predicting no sales ever.
print("A flat line at zero scores:", round(cost(0, 0), 2), "  <- terrible, as expected")
print("The (secretly) best line scores:", round(cost(3, 1.9), 4), "  <- much lower = much better")
print()
print("Slopes at the flat line:", [round(s, 2) for s in slopes(0, 0)])
print("Both NEGATIVE -> our line sits too LOW -> the math will push it UP.")

## 3. The loop — this *is* training

There is nothing else. Predict → measure the error → nudge both knobs downhill → repeat.

**The one subtlety that matters:** we work out where *both* knobs should go **before** moving either one
(that's what `temp0`/`temp1` are for). Both readings come from the same spot. That's called the
**simultaneous update** — the knobs move *together*, every single pass.

In [ ]:
ALPHA = 0.05        # how big a step we take downhill. Too big = chaos, too small = forever.
MAX_ITERS = 3000    # a safety cap so we can never loop forever
TOLERANCE = 1e-9    # "the score stopped improving" -> quit


def train(alpha=ALPHA, max_iters=MAX_ITERS, tolerance=TOLERANCE, verbose=True):
    """Walk downhill until the score stops improving. Returns the knobs + the history."""
    theta0, theta1 = 0.0, 0.0                     # start deliberately wrong: a flat line
    previous_cost = cost(theta0, theta1)
    history = [(0, theta0, theta1, previous_cost)]   # remember every step, so we can plot it

    for i in range(1, max_iters + 1):
        # STEP 1: feel the ground under both knobs, from where we stand right now
        slope0, slope1 = slopes(theta0, theta1)

        # STEP 2: work out where each knob SHOULD go - but don't move yet
        temp0 = theta0 - alpha * slope0
        temp1 = theta1 - alpha * slope1

        # STEP 3: now move BOTH knobs at the same instant (the simultaneous update)
        theta0, theta1 = temp0, temp1

        # Panic button: if the knobs fly off, alpha is too big (we'd crash otherwise)
        if abs(theta0) > 1e6 or abs(theta1) > 1e6:
            if verbose:
                print(f"{i:>5}  EXPLODED - alpha={alpha} is far too big. Steps leap over the valley.")
            return theta0, theta1, history

        current_cost = cost(theta0, theta1)
        history.append((i, theta0, theta1, current_cost))

        # Exit: the score barely moved, so more rounds buy nothing
        if abs(previous_cost - current_cost) < tolerance:
            if verbose:
                print(f"{i:>5} {theta0:>8.2f} {theta1:>8.2f} {current_cost:>10.4f}   STOP: converged")
            return theta0, theta1, history

        if verbose and i in {1, 2, 3, 5, 10, 30, 100, 500}:
            note = {10: "<- theta1 overshoots its peak...",
                    30: "<- ...easing back as theta0 catches up"}.get(i, "")
            print(f"{i:>5} {theta0:>8.2f} {theta1:>8.2f} {current_cost:>10.4f}   {note}")

        previous_cost = current_cost

    return theta0, theta1, history


print(f"{'iter':>5} {'theta0':>8} {'theta1':>8} {'cost J':>10}")
print("-" * 50)
print(f"{0:>5} {0.0:>8.2f} {0.0:>8.2f} {cost(0,0):>10.4f}   flat line - terrible, as intended")
learned0, learned1, history = train()

print(f"\nIt learned:  sales = {learned0:.4f} + {learned1:.4f} * spend")

## 4. Did it work? Grade it against the exact answer

Linear regression is one of the rare problems simple enough to solve **exactly** with algebra.
So we can mark our loop's homework.

*(Most models — every neural network — have **no** such formula. That's the whole reason gradient
descent exists.)*

In [ ]:
# The exact answer, straight from algebra (study doc, section 3.2).
x_bar = sum(ad_spend) / m                # average ad spend  = 2.5
y_bar = sum(sales) / m                   # average sales     = 7.75

numerator   = sum((x - x_bar) * (y - y_bar) for x, y in zip(ad_spend, sales))  # do they move together?
denominator = sum((x - x_bar) ** 2 for x in ad_spend)                          # how spread is spend?

true1 = numerator / denominator          # 9.5 / 5.0 = 1.9
true0 = y_bar - true1 * x_bar            # 7.75 - 1.9*2.5 = 3.0

print(f"  gradient descent walked to : {learned0:.4f} , {learned1:.4f}")
print(f"  algebra proves it is       : {true0:.4f} , {true1:.4f}")
print(f"  difference                 : {abs(learned0-true0):.2e} , {abs(learned1-true1):.2e}")
print("\n  It walked, blindfolded, to the same answer algebra proves. That's the point.")

## 5. Now *see* it — the line improving, step by step

Numbers in a table are abstract. Here's the same story as a picture.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(0.5, 4.5, 50)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

# LEFT: the line at various points in its education
for it, color, label in [(0, "#cccccc", "iter 0: flat, terrible"),
                         (1, "#fdbe85", "iter 1"),
                         (5, "#fd8d3c", "iter 5"),
                         (30, "#a63603", "iter 30 (overshot!)"),
                         (len(history) - 1, "#d62728", "final: best fit")]:
    _, t0, t1, _ = history[min(it, len(history) - 1)]
    ax1.plot(xs, t0 + t1 * xs, color=color, lw=2.5 if "final" in label else 1.6, label=label)
ax1.scatter(ad_spend, sales, s=90, zorder=5, color="#1f77b4", label="the data")
ax1.set_xlabel("Ad spend ($1,000s)"); ax1.set_ylabel("Weekly sales ($1,000s)")
ax1.set_title("The line teaching itself")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# RIGHT: the score falling - the "learning curve"
iters = [h[0] for h in history]
costs = [h[3] for h in history]
ax2.plot(iters, costs, color="#d62728", lw=2)
ax2.set_xlabel("iteration"); ax2.set_ylabel("cost J (badness)")
ax2.set_title("The learning curve: badness falling off a cliff")
ax2.set_xscale("symlog"); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("LEFT: notice 'iter 30' is STEEPER than the final line - it overshot, then came back.")
print("      That's the two knobs being coupled: theta1 raced ahead while theta0 lagged.")
print("RIGHT: the score drops every single step, even while the line looks like it's wandering.")

## 6. 🎛️ Your turn — break it

Drag the slider. **Predict what happens before you let go.**

- Push `alpha` **up** past ~0.4 → the steps get so big they *leap over* the valley and land higher on
  the far wall. Cost climbs instead of falling. It explodes.
- Drag it **down** to 0.001 → it crawls. It's still crawling when the safety cap fires.

There's no "correct" learning rate you can look up. This is why tuning it is a real job.

In [ ]:
from ipywidgets import interact, FloatLogSlider

def try_alpha(alpha=0.05):
    """Re-run the whole training with YOUR learning rate, and plot what happens."""
    t0, t1, hist = train(alpha=alpha, verbose=False)
    iters = [h[0] for h in hist]
    costs = [h[3] for h in hist]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    xs = np.linspace(0.5, 4.5, 50)
    ax1.scatter(ad_spend, sales, s=80, zorder=5, color="#1f77b4")
    if abs(t0) < 1e6 and abs(t1) < 1e6:
        ax1.plot(xs, t0 + t1 * xs, color="#d62728", lw=2.5)
        ax1.set_ylim(0, 13)
        ax1.set_title(f"Learned: sales = {t0:.2f} + {t1:.2f} * spend")
    else:
        ax1.set_title("EXPLODED - the line flew off the chart")
        ax1.set_ylim(0, 13)
    ax1.set_xlabel("Ad spend"); ax1.set_ylabel("Sales"); ax1.grid(alpha=0.3)

    ax2.plot(iters, costs, color="#d62728", lw=2)
    ax2.set_yscale("log"); ax2.set_xscale("symlog")
    ax2.set_xlabel("iteration"); ax2.set_ylabel("cost J (log scale)")
    ax2.set_title(f"alpha={alpha:g} | stopped after {len(hist)-1} steps | final cost {costs[-1]:.4g}")
    ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    if costs[-1] > costs[0]:
        print("The score went UP. Your steps are leaping over the valley. Alpha too big.")
    elif len(hist) - 1 >= MAX_ITERS:
        print("Hit the safety cap without converging - still crawling. Alpha too small.")
    else:
        print(f"Converged in {len(hist)-1} steps to {t0:.3f}, {t1:.3f}. (Truth: 3.0, 1.9)")

interact(try_alpha, alpha=FloatLogSlider(value=0.05, base=10, min=-3, max=-0.2, step=0.05,
                                         description="alpha"));

## 7. The scikit-learn version — and a surprise

In real work nobody hand-writes that loop. You call `.fit()`.

But `.fit()` is **not one thing**, and this trips up almost everyone 👇

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor

X = np.array([[1], [2], [3], [4]])      # sklearn wants a grid: one row per example
y = np.array([5, 7, 8, 11])

# --- The two lines that replace our hundred ---
model = LinearRegression()
model.fit(X, y)

print(f"LinearRegression found: theta0 = {model.intercept_:.10f}")
print(f"                        theta1 = {model.coef_[0]:.10f}")
print()
print("Look how EXACT that is. Ten decimal places. No walk downhill lands that cleanly.")
print("Because LinearRegression does NOT use gradient descent - it solves the algebra")
print("directly, exactly like our true0/true1 cell above. No loop. No learning rate.")

In [ ]:
# --- SGDRegressor: THIS one really does walk downhill ---
sgd = SGDRegressor(learning_rate="constant",
                   eta0=0.01,        # <- this IS our ALPHA
                   max_iter=1000,    # <- this IS our MAX_ITERS
                   tol=None,         # <- our TOLERANCE
                   penalty=None, random_state=0).fit(X, y)

sgd_more = SGDRegressor(learning_rate="constant", eta0=0.01, max_iter=50000,
                        tol=None, penalty=None, random_state=0).fit(X, y)

print(f"SGDRegressor (1,000 steps) : {sgd.intercept_[0]:.4f} , {sgd.coef_[0]:.4f}")
print(f"SGDRegressor (50,000 steps): {sgd_more.intercept_[0]:.4f} , {sgd_more.coef_[0]:.4f}")
print()
print("50x the work bought almost nothing! SGD reads ONE random row per step, so its")
print("slope is always a little wrong. It never settles - it jitters around the bottom")
print("forever. More steps = more jitter, not more accuracy. Smaller steps is the fix.")
print()
print(f"{'method':<28}{'theta0':>9}{'theta1':>9}   engine")
print("-" * 62)
print(f"{'our hand-written loop':<28}{learned0:>9.4f}{learned1:>9.4f}   batch gradient descent")
print(f"{'LinearRegression':<28}{model.intercept_:>9.4f}{model.coef_[0]:>9.4f}   exact algebra (NO descent)")
print(f"{'SGDRegressor':<28}{sgd.intercept_[0]:>9.4f}{sgd.coef_[0]:>9.4f}   stochastic gradient descent")
print(f"{'the truth':<28}{3.0:>9.4f}{1.9:>9.4f}   proved on paper")

## The takeaway

**You now know what every one of these means — because you wrote each one by hand:**

| our code | scikit-learn | what it is |
|---|---|---|
| `ALPHA = 0.05` | `eta0=0.05` | how big a step downhill |
| `MAX_ITERS = 3000` | `max_iter=3000` | the safety cap |
| `TOLERANCE = 1e-9` | `tol=1e-9` | "close enough, stop" |
| our whole `train()` loop | `.fit()` | the loop itself |

1. **Training is a loop.** Predict → measure error → nudge downhill → repeat. That's it.
2. **Both knobs move on every pass.** You never finish `theta0` and then start `theta1` — they're coupled
   (that's *why* `theta1` had to retreat from 2.55).
3. **`.fit()` is not one thing.** `LinearRegression` solves the algebra exactly; `SGDRegressor` actually
   walks. Same call, different engines.
4. **Exact algebra only exists for easy problems.** Every neural network on earth has no such shortcut —
   which is why gradient descent is *the* algorithm of modern AI, and why we hand-wrote it first.

**Next:** try it on a real dataset — open `World_Bank_Data_Foundations.ipynb`.